# Plagiarism Detection System: Complete Learning Guide

## Project Overview

This notebook demonstrates how to build a plagiarism detection system using three different approaches:

- **TF-IDF**: Traditional machine learning baseline
- **BiLSTM**: Deep learning with neural networks
- **BERT**: State-of-the-art transformer model

### Learning Objectives:

✓ Understand text preprocessing and cleaning  
✓ Learn how to split and prepare data for ML  
✓ Implement multiple plagiarism detection models  
✓ Compare model performance using evaluation metrics  
✓ Visualize and interpret results

### What You'll Learn:

Each section includes explanations of what the code does and why we do it this way.


In [1]:
# ===========================
# STEP 1: Import All Libraries
# ===========================

from math import pi
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix
)
from transformers import AutoTokenizer, BertForSequenceClassification
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim
import torch.nn as nn
import torch
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from collections import Counter
import re
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')


print("✅ All libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(
    f"Device available: {'GPU (CUDA)' if torch.cuda.is_available() else 'CPU'}")

✅ All libraries imported successfully!
PyTorch version: 2.10.0
Device available: CPU


## Configuration Section

**Why centralized configuration?**

- One place to change settings
- Easy to experiment with different parameters
- No hardcoded values scattered throughout
- Professional coding practice


In [2]:
# ========================================
# CONFIGURATION: All variables defined here
# ========================================
# Define all settings in one place for easy modification

# 🔧 RANDOM SEED (Reproducibility)
RANDOM_SEED = 42

# 📁 FILE PATHS
CSV_FILE_PATH = '/Users/abdurrahman/Downloads/test/trending_ai_papers.csv'
REPORT_PATH = '/Users/abdurrahman/Downloads/test/plagiarism_detection_report.txt'
METRIC_CHART_PATH = '/Users/abdurrahman/Downloads/test/metric_comparison.png'
RADAR_CHART_PATH = '/Users/abdurrahman/Downloads/test/radar_chart.png'
CONFUSION_MATRIX_PATH = '/Users/abdurrahman/Downloads/test/confusion_matrices.png'
ROC_CURVES_PATH = '/Users/abdurrahman/Downloads/test/roc_curves.png'

# 📊 DATA PROCESSING SETTINGS
DATASET_SIZE = 100  # Number of unique summaries to use (max 100)
TRAIN_SIZE = 0.70   # 70% for training
VAL_SIZE = 0.15     # 15% for validation
TEST_SIZE = 0.15    # 15% for testing

# 🔤 TF-IDF MODEL HYPERPARAMETERS
TFIDF_MAX_FEATURES = 5000      # Maximum features for TF-IDF
TFIDF_MIN_DF = 2               # Minimum document frequency
TFIDF_MAX_DF = 0.8             # Maximum document frequency
TFIDF_NGRAM_RANGE = (1, 2)     # Use unigrams and bigrams
TFIDF_SIMILARITY_THRESHOLD = 0.5  # Classification threshold

# 🧠 BiLSTM MODEL HYPERPARAMETERS
BILSTM_MAX_VOCAB = 10000       # Maximum vocabulary size
BILSTM_MAX_LEN = 100           # Maximum sequence length
BILSTM_EMBEDDING_DIM = 100     # Embedding dimension
BILSTM_HIDDEN_DIM = 64         # Hidden layer dimension
BILSTM_DROPOUT = 0.3           # Dropout rate
BILSTM_LEARNING_RATE = 0.001   # Adam optimizer learning rate
BILSTM_EPOCHS = 10             # Number of training epochs
BILSTM_BATCH_SIZE = 32         # Batch size

# ⭐ BERT MODEL HYPERPARAMETERS
BERT_MODEL_NAME = 'bert-base-uncased'  # Pre-trained BERT model
BERT_MAX_LEN = 128             # Maximum sequence length
BERT_LEARNING_RATE = 2e-5      # Learning rate (fine-tuning)
BERT_EPOCHS = 5                # Number of training epochs
BERT_BATCH_SIZE = 16           # Batch size

# 🎨 VISUALIZATION SETTINGS
CHART_FIGSIZE_SMALL = (10, 8)
CHART_FIGSIZE_MEDIUM = (15, 10)
CHART_FIGSIZE_LARGE = (18, 10)
COLORS = ['#3498db', '#e74c3c', '#2ecc71']  # Blue, Red, Green
MODEL_NAMES = ['TF-IDF', 'BiLSTM', 'BERT']
DPI = 150  # Resolution for saved images

# 📈 EVALUATION METRICS
METRICS_LIST = ['accuracy', 'precision', 'recall', 'f1_score', 'roc_auc']
EVALUATION_SETS = ['train', 'val', 'test']

# 🔍 DEVICE SETTINGS
DEVICE = torch.device('mps') if torch.backends.mps.is_available(
) else torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Set random seeds for reproducibility
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

print("✅ Configuration loaded!")
print(f"Dataset size: {DATASET_SIZE} summaries")
print(f"Split: {int(TRAIN_SIZE*100)}% train, {int(VAL_SIZE*100)}% val, {int(TEST_SIZE*100)}% test")
print(f"Using device: {DEVICE}")
print(f"Models to train: {', '.join(MODEL_NAMES)}")

✅ Configuration loaded!
Dataset size: 100 summaries
Split: 70% train, 15% val, 15% test
Using device: mps
Models to train: TF-IDF, BiLSTM, BERT


## STEP 2: Load and Explore Dataset

**What we're doing:**

- Load the data from a CSV file
- Examine the dataset structure
- Understand what data we're working with

**Why it matters:**
Before building models, we need to understand our data - its size, format, and content.


In [3]:
# Load the CSV file
df = pd.read_csv(CSV_FILE_PATH)

# Display basic information about the dataset
print("📊 DATASET INFORMATION")
print("=" * 50)
print(f"Number of rows: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")
print(f"\nColumn names: {df.columns.tolist()}")

# Show first few rows
print("\n📄 First 2 rows of data:")
print(df.head(2))

# Show data types
print("\n🔍 Data types:")
print(df.dtypes)

FileNotFoundError: [Errno 2] No such file or directory: '/Users/abdurrahman/Downloads/test/trending_ai_papers.csv'

In [ ]:
# ========================================
# CREATE PLAGIARISM DATASET FROM SUMMARIES
# ========================================

import random
random.seed(RANDOM_SEED)

print("\n📝 CREATING PLAGIARISM DATASET")
print("=" * 50)

summaries = df['Summary'].dropna().unique()[:DATASET_SIZE]
print(f"✓ Extracted {len(summaries)} unique paper summaries")


def create_paraphrased_text(text):
    """Create realistic paraphrases that are harder to detect"""
    import random
    random.seed(None)
    
    words = text.split()
    if len(words) < 3:
        return text
    
    # 1. Keep 60-85% of original words (more challenging)
    keep_ratio = random.uniform(0.60, 0.85)
    num_keep = max(3, int(len(words) * keep_ratio))
    
    # 2. Randomly select which words to keep (not just first half)
    indices_to_keep = sorted(random.sample(range(len(words)), num_keep))
    kept_words = [words[i] for i in indices_to_keep]
    
    # 3. Add some disorder - shuffle adjacent words occasionally
    for _ in range(max(1, len(kept_words) // 4)):
        if len(kept_words) > 1:
            idx = random.randint(0, len(kept_words) - 2)
            # Swap adjacent words with 40% probability
            if random.random() < 0.4:
                kept_words[idx], kept_words[idx + 1] = kept_words[idx + 1], kept_words[idx]
    
    # 4. Join and return
    result = ' '.join(kept_words) if kept_words else text
    return result


def create_similar_non_plagiarized(text1, text2):
    """Create more similar non-plagiarized pairs for harder classification"""
    import random
    random.seed(None)
    
    # Take parts from both texts and mix them
    words1 = text1.split()
    words2 = text2.split()
    
    # Use 60-75% from text1 and 25-40% from text2 (harder to distinguish)
    text1_ratio = random.uniform(0.60, 0.75)
    text1_count = max(3, int(len(words1) * text1_ratio))
    text2_count = max(2, int(len(words2) * 0.35))
    
    selected1 = words1[:text1_count] if text1_count <= len(words1) else words1
    selected2 = words2[:text2_count] if text2_count <= len(words2) else words2
    
    mixed = selected1 + selected2
    if len(mixed) > 0:
        random.shuffle(mixed)
        return ' '.join(mixed[:min(len(mixed), len(words1))])
    return text2


print("\n🔄 Generating text pairs with challenging paraphrases...")
text_pairs = []
labels = []

for i, summary in enumerate(summaries):
    # Create 2-3 plagiarized variants per summary (harder detection)
    num_variants = random.randint(2, 3)
    for _ in range(num_variants):
        paraphrased = create_paraphrased_text(summary)
        if paraphrased != summary:  # Only add if actually different
            text_pairs.append([summary, paraphrased])
            labels.append(1)
    
    # Create non-plagiarized pairs from different summaries
    if i + 1 < len(summaries):
        # Mix different summaries for harder non-plagiarized detection
        j = (i + random.randint(1, 3)) % len(summaries)
        if j != i:
            mixed = create_similar_non_plagiarized(summary, summaries[j])
            text_pairs.append([summary, mixed])
            labels.append(0)
    
    # Add some original non-plagiarized pairs
    if i + 2 < len(summaries):
        text_pairs.append([summary, summaries[i + 1]])
        labels.append(0)

plagiarism_df = pd.DataFrame({
    'source_text': [pair[0] for pair in text_pairs],
    'suspicious_text': [pair[1] for pair in text_pairs],
    'label': labels
})

print(f"\n✓ Dataset created successfully!")
print(f"  Total samples: {len(plagiarism_df)}")
print(f"\n📊 Label Distribution:")
label_counts = plagiarism_df['label'].value_counts()
print(f"  Non-Plagiarized (0): {label_counts[0]} samples")
print(f"  Plagiarized (1): {label_counts[1]} samples")

print(f"\n📌 Sample Plagiarized Pair (Label=1):")
sample_plagiarized = plagiarism_df[plagiarism_df['label'] == 1].iloc[0]
print(f"  Source:    {sample_plagiarized['source_text'][:80]}...")
print(f"  Suspicious: {sample_plagiarized['suspicious_text'][:80]}...")

print(f"\n📌 Sample Non-Plagiarized Pair (Label=0):")
sample_non_plagiarized = plagiarism_df[plagiarism_df['label'] == 0].iloc[0]
print(f"  Source:    {sample_non_plagiarized['source_text'][:80]}...")
print(f"  Suspicious: {sample_non_plagiarized['suspicious_text'][:80]}...")

In [ ]:
# ==============================
# EXPLORATORY DATA ANALYSIS (EDA)
# ==============================

print("\n📈 EXPLORATORY DATA ANALYSIS")
print("=" * 50)

plagiarism_df['source_length'] = plagiarism_df['source_text'].apply(
    lambda x: len(x.split())
)
plagiarism_df['suspicious_length'] = plagiarism_df['suspicious_text'].apply(
    lambda x: len(x.split())
)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Data Distribution Analysis', fontsize=14, fontweight='bold')

class_counts = plagiarism_df['label'].value_counts()
axes[0, 0].bar(['Non-Plagiarized (0)', 'Plagiarized (1)'],
               class_counts.values,
               color=['#3498db', '#e74c3c'], alpha=0.8)
axes[0, 0].set_title('Class Distribution', fontsize=11, fontweight='bold')
axes[0, 0].set_ylabel('Number of samples')
for i, v in enumerate(class_counts.values):
    axes[0, 0].text(i, v + 1, str(v), ha='center', fontweight='bold')

axes[0, 1].hist([plagiarism_df[plagiarism_df['label'] == 0]['source_length'],
                 plagiarism_df[plagiarism_df['label'] == 1]['source_length']],
                label=['Non-Plagiarized', 'Plagiarized'],
                bins=20, color=['#3498db', '#e74c3c'], alpha=0.7)
axes[0, 1].set_title('Source Text Lengths', fontsize=11, fontweight='bold')
axes[0, 1].set_xlabel('Number of words')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].legend()

axes[1, 0].hist([plagiarism_df[plagiarism_df['label'] == 0]['suspicious_length'],
                 plagiarism_df[plagiarism_df['label'] == 1]['suspicious_length']],
                label=['Non-Plagiarized', 'Plagiarized'],
                bins=20, color=['#3498db', '#e74c3c'], alpha=0.7)
axes[1, 0].set_title('Suspicious Text Lengths', fontsize=11, fontweight='bold')
axes[1, 0].set_xlabel('Number of words')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].legend()

axes[1, 1].axis('off')
stats_text = f"""
📊 STATISTICAL SUMMARY

Non-Plagiarized (Label=0):
  Source Text:
    • Average length: {plagiarism_df[plagiarism_df['label']==0]['source_length'].mean():.0f} words
    • Min: {plagiarism_df[plagiarism_df['label']==0]['source_length'].min()}, Max: {plagiarism_df[plagiarism_df['label']==0]['source_length'].max()}

  Suspicious Text:
    • Average length: {plagiarism_df[plagiarism_df['label']==0]['suspicious_length'].mean():.0f} words
    • Min: {plagiarism_df[plagiarism_df['label']==0]['suspicious_length'].min()}, Max: {plagiarism_df[plagiarism_df['label']==0]['suspicious_length'].max()}

Plagiarized (Label=1):
  Source Text:
    • Average length: {plagiarism_df[plagiarism_df['label']==1]['source_length'].mean():.0f} words
    • Min: {plagiarism_df[plagiarism_df['label']==1]['source_length'].min()}, Max: {plagiarism_df[plagiarism_df['label']==1]['source_length'].max()}

  Suspicious Text:
    • Average length: {plagiarism_df[plagiarism_df['label']==1]['suspicious_length'].mean():.0f} words
    • Min: {plagiarism_df[plagiarism_df['label']==1]['suspicious_length'].min()}, Max: {plagiarism_df[plagiarism_df['label']==1]['suspicious_length'].max()}
"""
axes[1, 1].text(0.05, 0.95, stats_text, fontsize=9, verticalalignment='top',
                family='monospace', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

print("\n✅ EDA Complete! Key findings:")
print(
    f"  • Dataset is balanced with {label_counts[0]} non-plagiarized and {label_counts[1]} plagiarized pairs")
print(
    f"  • Average text length: {plagiarism_df['source_length'].mean():.0f} words")
print(
    f"  • Text lengths range from {plagiarism_df['source_length'].min()} to {plagiarism_df['source_length'].max()} words")

## STEP 3: Data Preprocessing and Cleaning

**What we're doing:**

- Clean text by removing unwanted characters
- Normalize text to lowercase
- Remove URLs, emails, and special characters

**Why it matters:**

- Machine learning models work better with clean data
- Removes noise that can confuse the model
- Ensures consistent input format


In [ ]:
# ================================
# STEP 2: Text Preprocessing
# ================================

print("\n📝 TEXT PREPROCESSING")
print("=" * 50)


def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


print("✅ Preprocessing function created")

print("\nProcessing source texts...")
plagiarism_df['source_text_clean'] = plagiarism_df['source_text'].apply(
    preprocess_text)

print("Processing suspicious texts...")
plagiarism_df['suspicious_text_clean'] = plagiarism_df['suspicious_text'].apply(
    preprocess_text)

print("\n✅ Preprocessing complete!")

print("\n🔍 BEFORE vs AFTER CLEANING:")
print("-" * 50)
print("ORIGINAL TEXT (first 150 chars):")
print(f"  {plagiarism_df['source_text'].iloc[0][:150]}...")
print("\nCLEANED TEXT (first 150 chars):")
print(f"  {plagiarism_df['source_text_clean'].iloc[0][:150]}...")
print(
    f"\nReduction: {len(plagiarism_df['source_text'].iloc[0])} → {len(plagiarism_df['source_text_clean'].iloc[0])} characters")

## STEP 4: Train-Validation-Test Split (70/15/15)

**What we're doing:**

- Divide data into 3 sets:
  - **Training (70%)**: Used to train the model
  - **Validation (15%)**: Used to tune parameters
  - **Test (15%)**: Used to evaluate final performance

**Why it matters:**

- Prevents our model from "cheating" by memorizing test data
- Helps us make fair performance comparisons
- Ensures the model works on unseen data


In [ ]:
print("\n🔀 SPLITTING DATA INTO TRAIN/VALIDATION/TEST SETS")
print("=" * 50)

X_source = plagiarism_df['source_text_clean'].values
X_suspicious = plagiarism_df['suspicious_text_clean'].values
y = plagiarism_df['label'].values

print(f"Total samples: {len(y)}")

X_source_train, X_source_temp, X_suspicious_train, X_suspicious_temp, y_train, y_temp = train_test_split(
    X_source, X_suspicious, y,
    test_size=(1 - TRAIN_SIZE),
    random_state=RANDOM_SEED,
    stratify=y
)

X_source_val, X_source_test, X_suspicious_val, X_suspicious_test, y_val, y_test = train_test_split(
    X_source_temp, X_suspicious_temp, y_temp,
    test_size=0.50,
    random_state=RANDOM_SEED,
    stratify=y_temp
)

print("\n✅ Data split complete!")
print(f"\n📊 Dataset Sizes:")
print(
    f"  Training Set:   {len(X_source_train):4d} samples ({len(X_source_train)/len(X_source)*100:5.1f}%)")
print(
    f"  Validation Set: {len(X_source_val):4d} samples ({len(X_source_val)/len(X_source)*100:5.1f}%)")
print(
    f"  Test Set:       {len(X_source_test):4d} samples ({len(X_source_test)/len(X_source)*100:5.1f}%)")

print(f"\n📈 Class Distribution (checking balance):")
train_plagiarized = (y_train == 1).sum()
train_non_plagiarized = (y_train == 0).sum()
print(
    f"  Train - Non-Plagiarized: {train_non_plagiarized}, Plagiarized: {train_plagiarized}")

val_plagiarized = (y_val == 1).sum()
val_non_plagiarized = (y_val == 0).sum()
print(
    f"  Val   - Non-Plagiarized: {val_non_plagiarized}, Plagiarized: {val_plagiarized}")

test_plagiarized = (y_test == 1).sum()
test_non_plagiarized = (y_test == 0).sum()
print(
    f"  Test  - Non-Plagiarized: {test_non_plagiarized}, Plagiarized: {test_plagiarized}")

print("\n✅ Data is balanced across all sets! 🎉")

## STEP 5: TF-IDF Baseline Model

**What is TF-IDF?**

- TF (Term Frequency): How often a word appears in a document
- IDF (Inverse Document Frequency): How rare a word is across all documents
- Combines both to identify important words

**How it works for plagiarism:**

1. Convert text to numbers (TF-IDF vectors)
2. Compare similarity between source and suspicious text
3. If similarity > 0.5, mark as plagiarized

**Pros:** Fast, interpretable, good baseline  
**Cons:** Doesn't understand meaning, just word patterns


In [ ]:
print("\n🔤 TRAINING TF-IDF MODEL")
print("=" * 50)

print("\n1️⃣ Creating TF-IDF Vectorizer...")

combined_train_texts = list(X_source_train) + list(X_suspicious_train)

tfidf_vectorizer = TfidfVectorizer(
    max_features=TFIDF_MAX_FEATURES,
    min_df=TFIDF_MIN_DF,
    max_df=TFIDF_MAX_DF,
    ngram_range=TFIDF_NGRAM_RANGE
)

tfidf_vectorizer.fit(combined_train_texts)
print(f"✓ Vectorizer fitted on {len(combined_train_texts)} training texts")

print("\n2️⃣ Converting text to TF-IDF vectors...")

tfidf_train_source = tfidf_vectorizer.transform(X_source_train)
tfidf_train_suspicious = tfidf_vectorizer.transform(X_suspicious_train)

tfidf_val_source = tfidf_vectorizer.transform(X_source_val)
tfidf_val_suspicious = tfidf_vectorizer.transform(X_suspicious_val)

tfidf_test_source = tfidf_vectorizer.transform(X_source_test)
tfidf_test_suspicious = tfidf_vectorizer.transform(X_suspicious_test)

print(f"✓ Vectors created with {tfidf_train_source.shape[1]} features")

print("\n3️⃣ Computing similarity scores...")


def tfidf_predict(source_vec, suspicious_vec, threshold=TFIDF_SIMILARITY_THRESHOLD):
    similarities = []
    for s, u in zip(source_vec, suspicious_vec):
        sim = cosine_similarity(s, u)[0][0]
        similarities.append(sim)
    similarities = np.array(similarities)
    predictions = (similarities >= threshold).astype(int)
    return predictions, similarities


y_pred_train_tfidf, y_similarity_train = tfidf_predict(
    tfidf_train_source, tfidf_train_suspicious)
y_pred_val_tfidf, y_similarity_val = tfidf_predict(
    tfidf_val_source, tfidf_val_suspicious)
y_pred_test_tfidf, y_similarity_test = tfidf_predict(
    tfidf_test_source, tfidf_test_suspicious)

print(f"   ✓ Predictions made on all sets")


def calculate_metrics(y_true, y_pred, y_scores=None):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    auc = roc_auc_score(y_true, y_scores) if y_scores is not None else 0.0

    return {
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1_score': f1,
        'roc_auc': auc
    }


print("\n3️⃣ Calculating evaluation metrics...")

tfidf_metrics = {
    'train': calculate_metrics(y_train, y_pred_train_tfidf, y_similarity_train),
    'val': calculate_metrics(y_val, y_pred_val_tfidf, y_similarity_val),
    'test': calculate_metrics(y_test, y_pred_test_tfidf, y_similarity_test)
}

print("\n✅ TF-IDF Model Complete!")
print(f"\n📊 TF-IDF Performance on TEST set:")
print(f"   • Accuracy:  {tfidf_metrics['test']['accuracy']:.4f}")
print(f"   • Precision: {tfidf_metrics['test']['precision']:.4f}")
print(f"   • Recall:    {tfidf_metrics['test']['recall']:.4f}")
print(f"   • F1-Score:  {tfidf_metrics['test']['f1_score']:.4f}")
print(f"   • ROC-AUC:   {tfidf_metrics['test']['roc_auc']:.4f}")

## STEP 6: BiLSTM Deep Learning Model

**What is BiLSTM?**

- LSTM: Remembers important information from sequences
- **Bi**directional: Reads text left-to-right AND right-to-left
- Understands context and relationships between words

**How it works:**

1. Convert words to embeddings (dense vectors)
2. Pass through bidirectional LSTM layers
3. Concatenate information from both directions
4. Use fully connected layers to predict

**Pros:** Understands context and meaning  
**Cons:** Slower, needs more training data


In [ ]:
from collections import Counter

print("\n🧠 TRAINING BiLSTM MODEL")
print("=" * 50)

print("\n1️⃣ Building vocabulary...")


def build_vocab(texts, max_vocab=BILSTM_MAX_VOCAB):
    word_counter = Counter()
    for text in texts:
        words = text.split()
        word_counter.update(words)

    vocab = {}
    for idx, (word, _) in enumerate(word_counter.most_common(max_vocab - 2)):
        vocab[word] = idx + 2

    vocab['<PAD>'] = 0
    vocab['<UNK>'] = 1
    return vocab


vocab = build_vocab(list(X_source_train) + list(X_suspicious_train))
print(f"✓ Vocabulary created with {len(vocab)} unique words")

print("\n2️⃣ Converting text to number sequences...")


def text_to_sequence(texts, vocab, max_len=BILSTM_MAX_LEN):
    sequences = []
    for text in texts:
        sequence = [vocab.get(word, vocab['<UNK>']) for word in text.split()]
        sequence = sequence[:max_len]
        sequence += [vocab['<PAD>']] * (max_len - len(sequence))
        sequences.append(sequence)
    return np.array(sequences)


X_source_train_seq = text_to_sequence(X_source_train, vocab)
X_suspicious_train_seq = text_to_sequence(X_suspicious_train, vocab)
X_source_val_seq = text_to_sequence(X_source_val, vocab)
X_suspicious_val_seq = text_to_sequence(X_suspicious_val, vocab)
X_source_test_seq = text_to_sequence(X_source_test, vocab)
X_suspicious_test_seq = text_to_sequence(X_suspicious_test, vocab)

print(f"✓ All texts converted to sequences of length {BILSTM_MAX_LEN}")

print("\n3️⃣ Creating BiLSTM model architecture...")


class BiLSTMModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim=BILSTM_EMBEDDING_DIM,
                 hidden_dim=BILSTM_HIDDEN_DIM, output_dim=1):
        super(BiLSTMModel, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.fc1 = nn.Linear(hidden_dim * 2 * 2, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, output_dim)

        self.dropout = nn.Dropout(BILSTM_DROPOUT)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, source, suspicious):
        source_embed = self.embedding(source)
        suspicious_embed = self.embedding(suspicious)

        _, (source_hidden, _) = self.lstm(source_embed)
        _, (suspicious_hidden, _) = self.lstm(suspicious_embed)

        source_hidden = torch.cat([source_hidden[0], source_hidden[1]], dim=1)
        suspicious_hidden = torch.cat(
            [suspicious_hidden[0], suspicious_hidden[1]], dim=1)

        combined = torch.cat([source_hidden, suspicious_hidden], dim=1)

        x = self.dropout(self.relu(self.fc1(combined)))
        x = self.dropout(self.relu(self.fc2(x)))
        x = self.sigmoid(self.fc3(x))

        return x


bilstm_model = BiLSTMModel(len(vocab)).to(DEVICE)
optimizer = optim.Adam(bilstm_model.parameters(), lr=BILSTM_LEARNING_RATE)
criterion = nn.BCELoss()

print(f"✓ Model created. Using device: {DEVICE}")
print(
    f"  Total parameters: {sum(p.numel() for p in bilstm_model.parameters()):,}")

print("\n4️⃣ Defining training and evaluation functions...")


def train_bilstm_epoch(model, train_source, train_suspicious, train_labels, batch_size=BILSTM_BATCH_SIZE):
    model.train()
    total_loss = 0
    num_batches = 0

    for batch_idx in range(0, len(train_source), batch_size):
        source_batch = torch.LongTensor(
            train_source[batch_idx:batch_idx+batch_size]).to(DEVICE)
        suspicious_batch = torch.LongTensor(
            train_suspicious[batch_idx:batch_idx+batch_size]).to(DEVICE)
        labels_batch = torch.FloatTensor(
            train_labels[batch_idx:batch_idx+batch_size]).unsqueeze(1).to(DEVICE)

        optimizer.zero_grad()
        predictions = model(source_batch, suspicious_batch)
        loss = criterion(predictions, labels_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        num_batches += 1

    return total_loss / num_batches


def evaluate_bilstm(model, val_source, val_suspicious, val_labels):
    model.eval()
    all_predictions = []
    all_probabilities = []

    with torch.no_grad():
        for batch_idx in range(0, len(val_source), BILSTM_BATCH_SIZE):
            source_batch = torch.LongTensor(
                val_source[batch_idx:batch_idx+BILSTM_BATCH_SIZE]).to(DEVICE)
            suspicious_batch = torch.LongTensor(
                val_suspicious[batch_idx:batch_idx+BILSTM_BATCH_SIZE]).to(DEVICE)

            probabilities = model(
                source_batch, suspicious_batch).cpu().numpy().flatten()
            predictions = (probabilities >= 0.5).astype(int)

            all_probabilities.extend(probabilities)
            all_predictions.extend(predictions)

    return np.array(all_predictions), np.array(all_probabilities)


print(f"\n5️⃣ Training BiLSTM (this may take a minute)...")

for epoch in range(BILSTM_EPOCHS):
    loss = train_bilstm_epoch(
        bilstm_model, X_source_train_seq, X_suspicious_train_seq, y_train)
    if (epoch + 1) % 2 == 0:
        print(f"   Epoch {epoch+1:2d}/{BILSTM_EPOCHS} - Loss: {loss:.4f}")

print("✓ Training complete!")

print("\n6️⃣ Evaluating BiLSTM...")

y_pred_train_bilstm, y_prob_train_bilstm = evaluate_bilstm(
    bilstm_model, X_source_train_seq, X_suspicious_train_seq, y_train)
y_pred_val_bilstm, y_prob_val_bilstm = evaluate_bilstm(
    bilstm_model, X_source_val_seq, X_suspicious_val_seq, y_val)
y_pred_test_bilstm, y_prob_test_bilstm = evaluate_bilstm(
    bilstm_model, X_source_test_seq, X_suspicious_test_seq, y_test)

bilstm_metrics = {
    'train': calculate_metrics(y_train, y_pred_train_bilstm, y_prob_train_bilstm),
    'val': calculate_metrics(y_val, y_pred_val_bilstm, y_prob_val_bilstm),
    'test': calculate_metrics(y_test, y_pred_test_bilstm, y_prob_test_bilstm)
}

print("\n✅ BiLSTM Model Complete!")
print(f"\n📊 BiLSTM Performance on TEST set:")
print(f"   • Accuracy:  {bilstm_metrics['test']['accuracy']:.4f}")
print(f"   • Precision: {bilstm_metrics['test']['precision']:.4f}")
print(f"   • Recall:    {bilstm_metrics['test']['recall']:.4f}")
print(f"   • F1-Score:  {bilstm_metrics['test']['f1_score']:.4f}")
print(f"   • ROC-AUC:   {bilstm_metrics['test']['roc_auc']:.4f}")

## STEP 7: BERT Transformer Model

**What is BERT?**

- BERT: Bidirectional Encoder Representations from Transformers
- Pre-trained on massive amount of text
- Understands context and semantics (meaning)

**How it works:**

1. Tokenize text pair (special tokens combine both texts)
2. Use pre-trained BERT embeddings
3. Add classification head for plagiarism detection
4. Fine-tune on our data

**Pros:** State-of-the-art, understands context and meaning  
**Cons:** Large, slow, requires GPU for reasonable speed


In [ ]:
print("\n⭐ TRAINING BERT MODEL")
print("=" * 50)

print("\n1️⃣ Tokenizing texts...")

from transformers import BertTokenizer
tokenizer = BertTokenizer.from_pretrained(BERT_MODEL_NAME)


class BERTTextDataset(torch.utils.data.Dataset):
    def __init__(self, texts1, texts2, labels, tokenizer, max_len=BERT_MAX_LEN):
        self.texts1 = texts1
        self.texts2 = texts2
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        text1 = str(self.texts1[idx])
        text2 = str(self.texts2[idx])
        label = int(self.labels[idx])
        
        tokens1 = self.tokenizer.tokenize(text1)
        tokens2 = self.tokenizer.tokenize(text2)
        tokens = ['[CLS]'] + tokens1 + ['[SEP]'] + tokens2 + ['[SEP]']
        if len(tokens) > self.max_len:
            tokens = tokens[:self.max_len]
        
        input_ids = self.tokenizer.convert_tokens_to_ids(tokens)
        attention_mask = [1] * len(input_ids)
        pad_length = self.max_len - len(input_ids)
        
        if pad_length > 0:
            input_ids += [0] * pad_length
            attention_mask += [0] * pad_length

        return {
            'input_ids': torch.tensor(input_ids, dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
            'label': torch.tensor(label, dtype=torch.long)
        }


train_dataset = BERTTextDataset(X_source_train, X_suspicious_train, y_train, tokenizer)
val_dataset = BERTTextDataset(X_source_val, X_suspicious_val, y_val, tokenizer)
test_dataset = BERTTextDataset(X_source_test, X_suspicious_test, y_test, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=BERT_BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BERT_BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BERT_BATCH_SIZE)

print(f"✓ Data loaded with batch size {BERT_BATCH_SIZE}")

print("\n2️⃣ Loading BERT model...")

bert_model = BertForSequenceClassification.from_pretrained(BERT_MODEL_NAME, num_labels=2).to(DEVICE)
print(f"✓ BERT model loaded on {DEVICE}")

print("\n3️⃣ Defining training function...")


def train_bert_epoch(model, dataloader, optimizer):
    model.train()
    total_loss = 0

    for batch in dataloader:
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['label'].to(DEVICE)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        
        if isinstance(outputs, tuple):
            loss = outputs[0]
        else:
            loss = outputs.loss

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)


def evaluate_bert(model, dataloader):
    model.eval()
    all_predictions = []
    all_probabilities = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            
            if isinstance(outputs, tuple):
                logits = outputs[0]
            else:
                logits = outputs.logits

            probs = torch.softmax(logits, dim=1)
            predictions = torch.argmax(probs, dim=1).cpu().numpy()
            probabilities = probs[:, 1].cpu().numpy()

            all_predictions.extend(predictions)
            all_probabilities.extend(probabilities)

    return np.array(all_predictions), np.array(all_probabilities)


print("\n4️⃣ Setting up optimizer...")

bert_optimizer = optim.Adam(bert_model.parameters(), lr=BERT_LEARNING_RATE)

print(f"\n5️⃣ Training BERT (this may take 1-2 minutes)...")

for epoch in range(BERT_EPOCHS):
    loss = train_bert_epoch(bert_model, train_loader, bert_optimizer)
    print(f"   Epoch {epoch+1}/{BERT_EPOCHS} - Loss: {loss:.4f}")

print("✓ Training complete!")

print("\n6️⃣ Evaluating BERT...")

y_pred_train_bert, y_prob_train_bert = evaluate_bert(bert_model, train_loader)
y_pred_val_bert, y_prob_val_bert = evaluate_bert(bert_model, val_loader)
y_pred_test_bert, y_prob_test_bert = evaluate_bert(bert_model, test_loader)

bert_metrics = {
    'train': calculate_metrics(y_train, y_pred_train_bert, y_prob_train_bert),
    'val': calculate_metrics(y_val, y_pred_val_bert, y_prob_val_bert),
    'test': calculate_metrics(y_test, y_pred_test_bert, y_prob_test_bert)
}

print("\n✅ BERT Model Complete!")
print(f"\n📊 BERT Performance on TEST set:")
print(f"   • Accuracy:  {bert_metrics['test']['accuracy']:.4f}")
print(f"   • Precision: {bert_metrics['test']['precision']:.4f}")
print(f"   • Recall:    {bert_metrics['test']['recall']:.4f}")
print(f"   • F1-Score:  {bert_metrics['test']['f1_score']:.4f}")
print(f"   • ROC-AUC:   {bert_metrics['test']['roc_auc']:.4f}")


## STEP 8: Model Evaluation and Comparison

**What we're comparing:**

- All three models on the same test set
- Multiple evaluation metrics
- Strengths and weaknesses of each approach

**Key Metrics Explained:**

- **Accuracy**: % of correct predictions
- **Precision**: Of plagiarized predictions, how many are correct?
- **Recall**: Of actual plagiarisms, how many did we catch?
- **F1-Score**: Balance between precision and recall
- **ROC-AUC**: How well the model ranks plagiarized vs non-plagiarized


In [ ]:
print("\n" + "=" * 80)
print("📊 MODEL PERFORMANCE COMPARISON")
print("=" * 80)

metrics_dict = {
    MODEL_NAMES[0]: tfidf_metrics,
    MODEL_NAMES[1]: bilstm_metrics,
    MODEL_NAMES[2]: bert_metrics
}

print("\n🏆 FINAL TEST SET RESULTS:")
print("-" * 80)

comparison_data = []
for metric_name in METRICS_LIST:
    row = {'Metric': metric_name.upper().replace('_', ' ')}
    for model_name in MODEL_NAMES:
        value = metrics_dict[model_name]['test'][metric_name]
        row[model_name] = f"{value:.4f}"
    comparison_data.append(row)

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))

print("\n" + "=" * 80)
print("📈 DETAILED RESULTS (TEST SET)")
print("=" * 80)

for model_name in MODEL_NAMES:
    print(f"\n{model_name}:")
    print(
        f"  ├─ Accuracy:  {metrics_dict[model_name]['test']['accuracy']:.4f}")
    print(
        f"  ├─ Precision: {metrics_dict[model_name]['test']['precision']:.4f}")
    print(f"  ├─ Recall:    {metrics_dict[model_name]['test']['recall']:.4f}")
    print(
        f"  ├─ F1-Score:  {metrics_dict[model_name]['test']['f1_score']:.4f}")
    print(f"  └─ ROC-AUC:   {metrics_dict[model_name]['test']['roc_auc']:.4f}")

print("\n" + "=" * 80)
print("🏅 BEST PERFORMING MODELS:")
print("=" * 80)

best_accuracy = max([metrics_dict[m]['test']['accuracy'] for m in MODEL_NAMES])
best_f1 = max([metrics_dict[m]['test']['f1_score'] for m in MODEL_NAMES])

for model_name in MODEL_NAMES:
    if metrics_dict[model_name]['test']['accuracy'] == best_accuracy:
        print(f"\n🥇 Best Accuracy: {model_name} ({best_accuracy:.4f})")
    if metrics_dict[model_name]['test']['f1_score'] == best_f1:
        print(f"🥇 Best F1-Score: {model_name} ({best_f1:.4f})")

print("\n" + "=" * 80)

## STEP 9: Performance Visualization

**What we're visualizing:**

1. **Bar Charts**: Compare each metric across models
2. **Radar Chart**: Overview of all metrics for each model
3. **Confusion Matrices**: See true positive, false positive, etc.
4. **ROC Curves**: Model performance across different thresholds

**Why visualizations matter:**

- Easier to see patterns than raw numbers
- Helps identify where models excel or struggle
- Better for presentations and reports


In [ ]:
from math import pi

print("\n" + "=" * 80)
print("📊 CREATING VISUALIZATIONS")
print("=" * 80)

print("\n1️⃣ Creating metric comparison charts...")

fig, axes = plt.subplots(2, 3, figsize=CHART_FIGSIZE_LARGE)
fig.suptitle('Model Performance Comparison - Test Set',
             fontsize=16, fontweight='bold', y=1.00)

metrics_to_plot = ['accuracy', 'precision', 'recall', 'f1_score', 'roc_auc']

for idx, metric_name in enumerate(metrics_to_plot):
    row = idx // 3
    col = idx % 3

    values = [metrics_dict[model]['test'][metric_name]
              for model in MODEL_NAMES]

    bars = axes[row, col].bar(
        MODEL_NAMES, values, color=COLORS, alpha=0.8, edgecolor='black', linewidth=2)
    axes[row, col].set_title(metric_name.replace(
        '_', ' ').title(), fontsize=12, fontweight='bold')
    axes[row, col].set_ylabel('Score')
    axes[row, col].set_ylim([0, 1.05])
    axes[row, col].grid(axis='y', alpha=0.3, linestyle='--')

    for i, (bar, value) in enumerate(zip(bars, values)):
        height = bar.get_height()
        axes[row, col].text(i, height + 0.02, f'{value:.3f}',
                            ha='center', fontweight='bold', fontsize=10)

axes[1, 2].axis('off')

plt.tight_layout()
plt.savefig(METRIC_CHART_PATH, dpi=DPI, bbox_inches='tight')
plt.show()
print("✓ Metric comparison saved")

print("\n2️⃣ Creating radar chart...")

categories = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
num_categories = len(categories)

angles = [n / float(num_categories) * 2 * pi for n in range(num_categories)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=CHART_FIGSIZE_SMALL,
                       subplot_kw=dict(projection='polar'))

for model_idx, model_name in enumerate(MODEL_NAMES):
    values = [
        metrics_dict[model_name]['test']['accuracy'],
        metrics_dict[model_name]['test']['precision'],
        metrics_dict[model_name]['test']['recall'],
        metrics_dict[model_name]['test']['f1_score'],
        metrics_dict[model_name]['test']['roc_auc']
    ]
    values += values[:1]

    ax.plot(angles, values, 'o-', linewidth=2.5,
            label=model_name, color=COLORS[model_idx])
    ax.fill(angles, values, alpha=0.15, color=COLORS[model_idx])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, size=11)
ax.set_ylim(0, 1)
ax.set_title('Overall Model Performance\n(All Metrics at a Glance)',
             fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=11)
ax.grid(True, linestyle='--')

plt.tight_layout()
plt.savefig(RADAR_CHART_PATH, dpi=DPI, bbox_inches='tight')
plt.show()
print("✓ Radar chart saved")

print("\n3️⃣ Creating confusion matrices...")

fig, axes_cm = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Confusion Matrices - Test Set\n(TP=True Positive, FP=False Positive, TN=True Negative, FN=False Negative)',
             fontsize=12, fontweight='bold')

predictions = {
    'TF-IDF': y_pred_test_tfidf,
    'BiLSTM': y_pred_test_bilstm,
    'BERT': y_pred_test_bert
}

for idx, model_name in enumerate(MODEL_NAMES):
    cm = confusion_matrix(y_test, predictions[model_name])

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes_cm[idx], cbar=False,
                xticklabels=['Non-Plagiarized', 'Plagiarized'],
                yticklabels=['Non-Plagiarized', 'Plagiarized'])

    axes_cm[idx].set_title(f'{model_name}', fontweight='bold', fontsize=11)
    axes_cm[idx].set_ylabel('True Label')
    axes_cm[idx].set_xlabel('Predicted Label')

plt.tight_layout()
plt.savefig(CONFUSION_MATRIX_PATH, dpi=DPI, bbox_inches='tight')
plt.show()
print("✓ Confusion matrices saved")

print("\n4️⃣ Creating ROC curves...")

plt.figure(figsize=CHART_FIGSIZE_SMALL)

fpr_tfidf, tpr_tfidf, _ = roc_curve(y_test, y_similarity_test)
fpr_bilstm, tpr_bilstm, _ = roc_curve(y_test, y_prob_test_bilstm)
fpr_bert, tpr_bert, _ = roc_curve(y_test, y_prob_test_bert)

plt.plot(fpr_tfidf, tpr_tfidf,
         label=f'TF-IDF (AUC = {tfidf_metrics["test"]["roc_auc"]:.3f})',
         linewidth=2.5, color=COLORS[0])
plt.plot(fpr_bilstm, tpr_bilstm,
         label=f'BiLSTM (AUC = {bilstm_metrics["test"]["roc_auc"]:.3f})',
         linewidth=2.5, color=COLORS[1])
plt.plot(fpr_bert, tpr_bert,
         label=f'BERT (AUC = {bert_metrics["test"]["roc_auc"]:.3f})',
         linewidth=2.5, color=COLORS[2])

plt.plot([0, 1], [0, 1], 'k--', linewidth=1.5,
         label='Random Classifier (AUC = 0.500)')

plt.xlabel('False Positive Rate (1 - Specificity)',
           fontsize=12, fontweight='bold')
plt.ylabel('True Positive Rate (Sensitivity)', fontsize=12, fontweight='bold')
plt.title('ROC Curves - All Models\n(Higher curve = Better performance)',
          fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.grid(True, alpha=0.3, linestyle='--')
plt.xlim([0, 1])
plt.ylim([0, 1])

plt.tight_layout()
plt.savefig(ROC_CURVES_PATH, dpi=DPI, bbox_inches='tight')
plt.show()
print("✓ ROC curves saved")

print("\n✅ All visualizations created and saved!")
print("\n📁 Saved files:")
print(f"   • {METRIC_CHART_PATH}")
print(f"   • {RADAR_CHART_PATH}")
print(f"   • {CONFUSION_MATRIX_PATH}")
print(f"   • {ROC_CURVES_PATH}")

## STEP 10: Summary and Key Learnings

**What we built:**
A complete plagiarism detection system using three different approaches, from simple to advanced.

**Key Learning Outcomes:**
By completing this notebook, you've learned:

- How to preprocess and clean text data
- How to split data properly for machine learning
- How different models approach the same problem
- How to evaluate and compare models
- How to visualize results for interpretation


In [ ]:
print("\n" + "=" * 100)
print(" " * 20 + "🎓 PLAGIARISM DETECTION SYSTEM - COMPLETE ANALYSIS")
print("=" * 100)

summary_report = f"""
{'='*100}
SECTION 1: PROJECT OVERVIEW
{'='*100}

📊 Dataset Information:
   • Total samples: {len(plagiarism_df)}
   • Plagiarized pairs (label=1): {(y == 1).sum()}
   • Non-plagiarized pairs (label=0): {(y == 0).sum()}
   • Train/Val/Test split: {int(TRAIN_SIZE*100)}% / {int(VAL_SIZE*100)}% / {int(TEST_SIZE*100)}%
   • Average text length: {plagiarism_df['source_length'].mean():.0f} words

{'='*100}
SECTION 2: MODEL COMPARISON (TEST SET)
{'='*100}

TF-IDF Model (Traditional ML):
   ✓ Accuracy:  {tfidf_metrics['test']['accuracy']:.4f}
   ✓ Precision: {tfidf_metrics['test']['precision']:.4f}
   ✓ Recall:    {tfidf_metrics['test']['recall']:.4f}
   ✓ F1-Score:  {tfidf_metrics['test']['f1_score']:.4f}
   ✓ ROC-AUC:   {tfidf_metrics['test']['roc_auc']:.4f}

BiLSTM Model (Deep Learning):
   ✓ Accuracy:  {bilstm_metrics['test']['accuracy']:.4f}
   ✓ Precision: {bilstm_metrics['test']['precision']:.4f}
   ✓ Recall:    {bilstm_metrics['test']['recall']:.4f}
   ✓ F1-Score:  {bilstm_metrics['test']['f1_score']:.4f}
   ✓ ROC-AUC:   {bilstm_metrics['test']['roc_auc']:.4f}

BERT Model (Transformer-based):
   ✓ Accuracy:  {bert_metrics['test']['accuracy']:.4f}
   ✓ Precision: {bert_metrics['test']['precision']:.4f}
   ✓ Recall:    {bert_metrics['test']['recall']:.4f}
   ✓ F1-Score:  {bert_metrics['test']['f1_score']:.4f}
   ✓ ROC-AUC:   {bert_metrics['test']['roc_auc']:.4f}

{'='*100}
SECTION 3: MODEL COMPARISON - PROS AND CONS
{'='*100}

🔤 TF-IDF (Baseline - Traditional Machine Learning)
   What it does:
      - Converts text to word frequencies
      - Compares similarity using cosine distance
      - Simple and interpretable

   Pros:
      ✓ Very fast (inference in milliseconds)
      ✓ Easy to understand and explain
      ✓ Works with limited data
      ✓ Low memory requirements
      ✓ Good baseline for comparison

   Cons:
      ✗ Doesn't understand word meaning (semantic)
      ✗ Just counts word patterns
      ✗ Poor at detecting paraphrased plagiarism
      ✗ Limited context understanding

   When to use:
      → Production systems with speed requirements
      → Embedded or edge device deployment
      → Quick prototyping and baselines

🧠 BiLSTM (Deep Learning with Recurrent Neural Networks)
   What it does:
      - Learns word embeddings (word meaning)
      - Processes text both forward and backward
      - Understands sequential dependencies

   Pros:
      ✓ Better at capturing context
      ✓ Understands word relationships
      ✓ Good performance on moderate datasets
      ✓ Smaller than BERT

   Cons:
      ✗ Slower training and inference than TF-IDF
      ✗ Requires more data
      ✗ Less robust than transformers
      ✗ Harder to tune hyperparameters

   When to use:
      → Balanced projects (good performance, reasonable resources)
      → When you have 5000-50000 training samples
      → Medium computational resources available

⭐ BERT (Transformer-based - State-of-the-art)
   What it does:
      - Pre-trained on massive text corpus
      - Bidirectional context understanding
      - Fine-tunes for specific tasks

   Pros:
      ✓ Best semantic understanding
      ✓ State-of-the-art performance
      ✓ Excellent at detecting paraphrased plagiarism
      ✓ Pre-trained (transfer learning)

   Cons:
      ✗ Slow inference (seconds per prediction)
      ✗ Large memory requirements (>1GB)
      ✗ Needs GPU for reasonable speed
      ✗ Expensive to deploy

   When to use:
      → Maximum accuracy needed
      → Resources available (GPU, memory)
      → Large or complex text datasets
      → Production system with offline processing

{'='*100}
SECTION 4: WHICH MODEL SHOULD YOU USE?
{'='*100}

Choose TF-IDF if:
   → You need fast predictions (< 100ms)
   → You have limited computational resources
   → You want an interpretable model
   → You need a quick baseline

Choose BiLSTM if:
   → You want better than TF-IDF but faster than BERT
   → You have GPU available but not abundant
   → You have 5,000-50,000 training examples
   → You need balance between accuracy and speed

Choose BERT if:
   → Accuracy is your top priority
   → You have GPU/cloud resources
   → You can process in batches (not real-time)
   → You're building a production system that needs best performance

{'='*100}
SECTION 5: KEY CONCEPTS YOU LEARNED
{'='*100}

1. TEXT PREPROCESSING
   • Why: Machine learning models need clean input
   • What: Lowercasing, removing URLs, removing special chars
   • Result: Consistent, normalized text data

2. DATA SPLITTING
   • Why: Prevent overfitting and get honest performance estimates
   • How: 70% train, 15% validation, 15% test
   • Benefit: Fair model comparison and evaluation

3. VECTORIZATION (TF-IDF)
   • What: Converting text to numbers (vectors)
   • How: Based on word frequency and importance
   • Use: Measures similarity via cosine distance

4. EMBEDDINGS (BiLSTM)
   • What: Dense vectors that capture word meaning
   • How: Learned during neural network training
   • Benefit: Semantic understanding vs. just word counts

5. TRANSFER LEARNING (BERT)
   • What: Using pre-trained models for new tasks
   • Why: Don't need to train from scratch
   • How: Fine-tune on your specific data

6. EVALUATION METRICS
   • Accuracy: Overall correctness
   • Precision: When we say "plagiarized", how often right?
   • Recall: Of actual plagiarisms, how many do we catch?
   • F1: Balance between precision and recall
   • ROC-AUC: How well the model separates classes

{'='*100}
SECTION 6: NEXT STEPS FOR IMPROVEMENT
{'='*100}

Short-term improvements:
   1. Add cross-validation for more robust evaluation
   2. Hyperparameter tuning (learning rate, batch size, etc.)
   3. Try other models (XGBoost, GPT, etc.)
   4. Data augmentation (create synthetic examples)
   5. Ensemble methods (combine all three models)

Long-term projects:
   1. Real plagiarism detection on academic papers
   2. Deploy to production (Flask/FastAPI)
   3. Real-time detection API
   4. Domain-specific fine-tuning
   5. Multilingual plagiarism detection

Advanced techniques:
   1. Semantic similarity models (Sentence-BERT)
   2. Large language models (GPT-3.5, GPT-4)
   3. Few-shot learning
   4. Active learning for data acquisition
   5. Explainability (what words triggered detection?)

{'='*100}
SECTION 7: COMMON MISTAKES TO AVOID
{'='*100}

❌ Data Leakage:
   Problem: Test data information leaks into training
   Solution: Always split BEFORE preprocessing when possible

❌ Imbalanced Evaluation:
   Problem: Using accuracy for imbalanced datasets
   Solution: Use precision, recall, F1-score, and ROC-AUC

❌ Overfitting:
   Problem: Model memorizes training data
   Solution: Use validation set, early stopping, regularization

❌ Ignoring Baseline:
   Problem: Not comparing to simple models
   Solution: Always include TF-IDF or similar baseline

❌ Hyperparameter Overfitting:
   Problem: Tuning on test set
   Solution: Use validation set for tuning, test only for final eval

{'='*100}
FINAL THOUGHTS
{'='*100}

Congratulations! You've built a complete machine learning system from scratch.

Key takeaways:
   ✓ There's no single "best" model - it depends on your constraints
   ✓ Simple models (TF-IDF) are often underrated and useful baselines
   ✓ Deep learning and transformers are powerful but require resources
   ✓ Evaluation and visualization are as important as the model itself
   ✓ Always start simple and add complexity as needed

Remember: The best model is the one that solves your actual problem
           within your constraints (speed, memory, latency, cost).

Happy learning! 🎉

{'='*100}
"""

print(summary_report)

with open(REPORT_PATH, 'w') as f:
    f.write(summary_report)

print(f"\n📄 Full report saved to: {REPORT_PATH}")
print("\n✅ ANALYSIS COMPLETE! You now understand plagiarism detection systems! 🎓")